# ML-5: Unsupervised Anomaly Detection using Isolation Forest

This notebook implements an **unsupervised anomaly detection pipeline** using **Isolation Forest** to identify unusual machine operating conditions.

## Key Objectives & Requirements:
1. **No Data Leakage**: Exclude identifier columns (`UDI`, `Product ID`) and failure-mode columns (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) from model inputs.
2. **Target Exclusion during Training**: The `Machine failure` label is **never** used as a model feature or training target.
3. **Normal-Operating Baseline Strategy**: The Isolation Forest is trained strictly on normal operating records (`Machine failure == 0`) to model baseline machine operational parameters.
4. **Domain Preprocessing**: Uses `DomainFeatureEngineer` to compute physical domain features ($\Delta T$, $P_{\text{mech}}$, Overstrain Index) and `ColumnTransformer` for feature scaling and encoding.
5. **Output Specification**: Produces decision anomaly score (`anomaly_score`) and boolean flag (`is_anomaly`).
6. **Contamination & Threshold Selection**: Configured with explicit contamination rate (`contamination = 0.035`, i.e., ~3.5%) to reflect operational anomaly expectations.
7. **Conceptual Distinction**: Explicitly states **Anomaly $\neq$ Failure**.
8. **Sanity Checking**: Uses actual failure labels solely post-training to inspect detected anomaly rates across normal vs. failed operational states.
9. **Pipeline Artifact Export**: Exports the reusable anomaly detection pipeline to `ml/models/anomaly_pipeline.joblib`.

In [2]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

# Append src module to system path for reusable feature engineering functions
sys.path.append(os.path.abspath("../src"))
from feature_engineering import DomainFeatureEngineer, build_preprocessing_pipeline

Matplotlib is building the font cache; this may take a moment.


## 1. Load Data & Isolation Forest Strategy

We ingest the cleaned dataset, exclude leakage-prone failure mode columns, and separate **normal operating records** (`Machine failure == 0`) for training the Isolation Forest model.

In [3]:
# Load dataset
data_path = "../../data/cleaned_predictive_maintenance.csv"
if not os.path.exists(data_path):
    data_path = "../../data/ai4i2020.csv"

df = pd.read_csv(data_path)
print(f"Loaded dataset shape: {df.shape}")

# Define feature columns and excluded columns
target_col = 'Machine failure'
excluded_cols = ['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
feature_cols = [c for c in df.columns if c not in excluded_cols and c != target_col]

print(f"Agreed operating input features ({len(feature_cols)}): {feature_cols}")

# Separate features (X) and target (y)
X_all = df[feature_cols].copy()
y_all = df[target_col].copy()

# Training Subset: Train strictly on NORMAL operating records (Machine failure == 0)
normal_mask = (y_all == 0)
X_train_normal = X_all[normal_mask].copy()

print(f"Total Records: {len(X_all)}")
print(f"Normal Operating Records for Training (Machine failure == 0): {len(X_train_normal)}")
print(f"Failed Records reserved for Sanity Check ONLY (Machine failure == 1): {(~normal_mask).sum()}")

Loaded dataset shape: (100, 7)
Agreed operating input features (6): ['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
Total Records: 100
Normal Operating Records for Training (Machine failure == 0): 97
Failed Records reserved for Sanity Check ONLY (Machine failure == 1): 3


## 2. Feature Engineering & Preprocessing

We apply physical domain feature engineering (`temperature_difference`, `mechanical_power_W`, `overstrain_index`) and fit the `ColumnTransformer` **strictly on normal operating training data**.

In [4]:
# Apply domain feature engineering
engineer = DomainFeatureEngineer()
X_eng_normal = engineer.transform(X_train_normal)
X_eng_all = engineer.transform(X_all)

# Identify categorical and numerical columns
categorical_cols = ['Type']
numerical_cols = [c for c in X_eng_normal.columns if c not in categorical_cols]

# Build preprocessing pipeline (OneHotEncoder + StandardScaler)
preprocessor = build_preprocessing_pipeline(categorical_cols, numerical_cols)

# Fit preprocessor strictly on normal operating data
X_proc_normal = preprocessor.fit_transform(X_eng_normal)
X_proc_all = preprocessor.transform(X_eng_all)

# Get recoverable feature names
feature_names = list(preprocessor.get_feature_names_out())
X_train_proc_df = pd.DataFrame(X_proc_normal, columns=feature_names)
X_all_proc_df = pd.DataFrame(X_proc_all, columns=feature_names)

print(f"Processed Feature Shape: {X_all_proc_df.shape}")
print("Processed Feature Names:")
for name in feature_names:
    print(f"  • {name}")

Processed Feature Shape: (100, 11)
Processed Feature Names:
  • Type_H
  • Type_L
  • Type_M
  • Air temperature [K]
  • Process temperature [K]
  • Rotational speed [rpm]
  • Torque [Nm]
  • Tool wear [min]
  • temperature_difference
  • mechanical_power_W
  • overstrain_index


## 3. Train Isolation Forest Model

We initialize and train scikit-learn's `IsolationForest`.

### Parameters:
- `n_estimators=100`: Number of isolation trees in ensemble.
- `contamination=0.035`: Expected baseline operational anomaly fraction (~3.5%).
- `random_state=42`: Ensures deterministic and reproducible tree splits.

### Anomaly Score & Decision Boundary:
- `decision_function(X)`: Returns decision score where lower/negative values indicate anomalies, and positive values indicate normal operation.
- `is_anomaly`: Boolean `True` if `decision_function < 0.0` (i.e. model `predict(X) == -1`), otherwise `False`.

In [5]:
# Initialize and fit Isolation Forest on processed normal operating records
contamination_setting = 0.035
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=contamination_setting,
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_train_proc_df)
print("Isolation Forest successfully trained on normal operating baseline!")

# Compute predictions and scores across the FULL dataset
# decision_function: < 0 => anomaly, >= 0 => normal
raw_scores = iso_forest.decision_function(X_all_proc_df)
predictions = iso_forest.predict(X_all_proc_df) # -1: anomaly, 1: inlier

# Format final output columns
# anomaly_score: raw decision score (lower = more anomalous)
# is_anomaly: True if predicted as anomaly (-1)
df['anomaly_score'] = raw_scores
df['is_anomaly'] = (predictions == -1)

anomaly_count = df['is_anomaly'].sum()
print(f"\nTotal Detected Operational Anomalies: {anomaly_count} / {len(df)} ({anomaly_count / len(df):.2%})")
print("Sample Anomaly Outputs:")
print(df[['Type', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'anomaly_score', 'is_anomaly']].head(10))

Isolation Forest successfully trained on normal operating baseline!

Total Detected Operational Anomalies: 5 / 100 (5.00%)
Sample Anomaly Outputs:
  Type  Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  anomaly_score  \
0    M                    1551         42.8                0      -0.000817   
1    L                    1408         46.3                3       0.061999   
2    L                    1498         49.4                5       0.014844   
3    L                    1433         39.5                7       0.060375   
4    L                    1408         40.0                9       0.069113   
5    M                    1425         41.9               11       0.014961   
6    L                    1558         42.4               14       0.053018   
7    L                    1527         40.2               16       0.064003   
8    M                    1667         28.6               18       0.025238   
9    M                    1741         28.0               21   

## 4. Conceptual Clarification: Anomaly $\neq$ Failure & Post-Training Sanity Check

> ### ⚠️ Critical Architectural Note: Anomaly $\neq$ Failure
> - **Anomaly**: Indicates an unusual or extreme machine operating state (e.g. process temperature spike, abnormal rotational speed/torque combination) that deviates statistically from normal baseline operation.
> - **Machine Failure**: Represents physical equipment breakdown resulting from cumulative tool wear, overheating (HDF), power overload (PWF), tool wear failure (TWF), or overstrain (OSF).
> - **Relationship**: An anomaly does **not** guarantee an immediate failure, nor do all failures begin with immediate extreme parameter anomalies. Anomaly detection provides an independent risk signal for proactive monitoring.

### Post-Training Sanity Check
We evaluate how detected anomalies correlate with actual `Machine failure` labels (used strictly post-training for verification).

In [6]:
# Cross-tabulation between actual Machine failure and detected is_anomaly
ctable = pd.crosstab(
    df['Machine failure'], 
    df['is_anomaly'], 
    rownames=['Actual Failure'], 
    colnames=['Is Anomaly']
)
print("Cross-Tabulation Matrix (Actual Failure vs. Detected Anomaly):")
print(ctable)

# Calculate anomaly rates per group
normal_anomaly_rate = df[df['Machine failure'] == 0]['is_anomaly'].mean()
failure_anomaly_rate = df[df['Machine failure'] == 1]['is_anomaly'].mean()

print(f"\nAnomaly Rate in Normal Operating Records (y=0): {normal_anomaly_rate:.2%}")
print(f"Anomaly Rate in Failed Operating Records (y=1): {failure_anomaly_rate:.2%}")

# Summary of feature means: Normal vs Anomaly
feature_summary = df.groupby('is_anomaly')[['Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Process temperature [K]']].mean()
print("\nFeature Averages by Anomaly Status:")
print(feature_summary)

Cross-Tabulation Matrix (Actual Failure vs. Detected Anomaly):
Is Anomaly      False  True 
Actual Failure              
0                  93      4
1                   2      1

Anomaly Rate in Normal Operating Records (y=0): 4.12%
Anomaly Rate in Failed Operating Records (y=1): 33.33%

Feature Averages by Anomaly Status:
            Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  \
is_anomaly                                                         
False                  1520.884211    40.371579        91.663158   
True                   2008.000000    23.100000        56.800000   

            Process temperature [K]  
is_anomaly                           
False                    309.047368  
True                     309.040000  


## 5. Export Reusable Anomaly Pipeline Package

We bundle the domain feature engineer, preprocessor, Isolation Forest model, feature schema, contamination setting, and threshold into a reusable `.joblib` package for Backend API consumption.

In [7]:
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

anomaly_pipeline_bundle = {
    "engineer": engineer,
    "preprocessor": preprocessor,
    "model": iso_forest,
    "feature_names": feature_names,
    "input_features": feature_cols,
    "contamination": contamination_setting,
    "threshold": 0.0,
    "model_type": "IsolationForest",
    "description": "Isolation Forest Anomaly Detection Pipeline trained on normal operating baseline records."
}

artifact_path = os.path.join(models_dir, "anomaly_pipeline.joblib")
joblib.dump(anomaly_pipeline_bundle, artifact_path)
print(f"Saved reusable anomaly pipeline package to: {os.path.abspath(artifact_path)}")

# Verification load test
loaded_bundle = joblib.load(artifact_path)
print("\nVerification Test on Loaded Package:")
sample_input = X_all.iloc[:2]
sample_eng = loaded_bundle["engineer"].transform(sample_input)
sample_proc = loaded_bundle["preprocessor"].transform(sample_eng)
sample_scores = loaded_bundle["model"].decision_function(sample_proc)
sample_is_anomaly = (loaded_bundle["model"].predict(sample_proc) == -1)

print("Sample Predictions from Loaded Artifact:")
print(f"  Anomaly Scores: {sample_scores}")
print(f"  Is Anomaly:     {sample_is_anomaly}")

Saved reusable anomaly pipeline package to: c:\Users\bingu\OneDrive\Desktop\cognizant\predictive-maintenance\ml\models\anomaly_pipeline.joblib

Verification Test on Loaded Package:
Sample Predictions from Loaded Artifact:
  Anomaly Scores: [-0.00081675  0.06199887]
  Is Anomaly:     [ True False]


C:\Users\bingu\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
C:\Users\bingu\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
